# Lab 1A: API fundamentals
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 1 · about 30 minutes**

**Objectives**
- Call the Messages API directly over REST, then through the Python SDK
- Put model ID and limits in one config module
- Keep conversation history in your own code (the API is stateless)
- Detect and handle max_tokens truncation

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic requests
import os, json, time, random, importlib
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Part 1: the raw REST request
The SDK is a thin layer over this HTTP call. The terminal equivalent is:
```bash
curl https://api.anthropic.com/v1/messages \
  -H "x-api-key: $ANTHROPIC_API_KEY" \
  -H "anthropic-version: 2023-06-01" \
  -H "content-type: application/json" \
  -d '{"model": "claude-sonnet-5", "max_tokens": 300, "messages": [{"role": "user", "content": "..."}]}'
```
**Predict:** which three headers are required, and what happens if one is missing?

In [ ]:
import requests
TICKET = ("Ticket T-77. Customer Asha (account C-1001) reports her fiber router drops every evening "
          "between 7pm and 9pm since 20 September. She has restarted it twice. Plan: Fiber 300.")
r = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"],
             "anthropic-version": "2023-06-01",
             "content-type": "application/json"},
    json={"model": "claude-sonnet-5", "max_tokens": 300,
          "messages": [{"role": "user", "content": "Summarize this support ticket in two sentences:\n" + TICKET}]},
    timeout=60)
print("HTTP", r.status_code)
body = r.json()
print(json.dumps(body, indent=2)[:1200])

In [ ]:
# Read the fields you will check on every call
if r.ok:
    print("stop_reason:", body["stop_reason"])
    print("usage:", body["usage"])
    print("text:", body["content"][0]["text"])
else:
    print("error type:", body["error"]["type"], "|", body["error"]["message"])

Try it: remove the `anthropic-version` header and rerun the request cell. Read the error type, then put the header back.

## Part 2: the same call through the Python SDK
The SDK reads `ANTHROPIC_API_KEY` from the environment, sets the headers, parses the response into typed objects, and retries some transient failures automatically.

In [ ]:
sdk_client = anthropic.Anthropic()
resp = sdk_client.messages.create(
    model="claude-sonnet-5", max_tokens=300,
    messages=[{"role": "user", "content": "Summarize this support ticket in two sentences:\n" + TICKET}])
print("stop_reason:", resp.stop_reason)
print("usage:", resp.usage)
print("text:", resp.content[0].text)

In [ ]:
# Same shape as the REST JSON, as typed objects
print(resp.model_dump_json(indent=2)[:1200])

## Part 3: one config module
The model ID above is typed twice already. In a real service it would spread across many files, and a model change would surprise you. Put settings in one place, pin the model, and keep the key in the environment.

In [ ]:
%%writefile config.py
"""One place for Claude settings. Code imports from here; nothing is hard-coded elsewhere."""
import os

# Pin the exact model ID your tests passed on. Confirm current IDs on the models overview page:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
MAX_TOKENS = 1024
MAX_RETRIES = 2        # one retry layer: the SDK's own
TIMEOUT_S = 60
API_VERSION = "2023-06-01"
# The key is NEVER stored here. It comes from the environment: ANTHROPIC_API_KEY

In [ ]:
import config
importlib.reload(config)          # pick up edits if you rerun the cell above
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    """Print the parts of a response you check on every call."""
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

print("model:", config.MODEL, "| max_retries:", config.MAX_RETRIES, "| timeout:", config.TIMEOUT_S, "s")

In [ ]:
resp = client.messages.create(
    model=config.MODEL, max_tokens=config.MAX_TOKENS,
    system="You summarize telecom support tickets for engineers. Two sentences, no greeting.",
    messages=[{"role": "user", "content": TICKET}])
log_response(resp, "[config + system prompt]")

Checkpoint: search this notebook for the model ID. From here on it should appear only in `config.py`.

## Part 4: the API is stateless
Your code owns the conversation. A helper that keeps history and resends it every call:

In [ ]:
class Chat:
    def __init__(self, system=None):
        self.system, self.history = system, []
    def send(self, text):
        self.history.append({"role": "user", "content": text})
        kw = dict(model=config.MODEL, max_tokens=config.MAX_TOKENS, messages=self.history)
        if self.system:
            kw["system"] = self.system
        resp = client.messages.create(**kw)
        self.history.append({"role": "assistant", "content": resp.content})   # keep Claude's turn too
        log_response(resp, f"[turn {len(self.history) // 2}]")
        return resp

chat = Chat(system="You are a concise telecom billing assistant.")
chat.send("My account is C-1001 and my September bill was 74.99 USD, charged twice.")
chat.send("Which account did I give you, and how much was I overcharged?")

Now drop the history and ask the same follow-up. **Predict** what happens before you run it.

In [ ]:
lost = client.messages.create(model=config.MODEL, max_tokens=200,
    messages=[{"role": "user", "content": "Which account did I give you, and how much was I overcharged?"}])
log_response(lost, "[no history]")
print("\nMessages the helper sent on its last call:", len(chat.history) - 1)

## Part 5: truncation is not an answer
`stop_reason == "max_tokens"` means the output hit the cap and is incomplete. Here a JSON response is cut off.

In [ ]:
ask = [{"role": "user", "content": "Return a JSON array of 15 objects, each with fields ticket_id, category, priority, "
                                   "for synthetic telecom tickets. Return only the JSON."}]
short = client.messages.create(model=config.MODEL, max_tokens=80, messages=ask)
print("stop_reason:", short.stop_reason)
try:
    json.loads(short.content[0].text)
    print("parsed")
except json.JSONDecodeError as e:
    print("JSON parse failed:", e)

In [ ]:
def create_complete(messages, max_tokens=256, ceiling=4096, **kw):
    """Never return truncated output: raise the cap and retry until complete or the ceiling is hit."""
    while True:
        resp = client.messages.create(model=config.MODEL, max_tokens=max_tokens, messages=messages, **kw)
        print(f"  max_tokens={max_tokens} -> stop_reason={resp.stop_reason}")
        if resp.stop_reason != "max_tokens":
            return resp
        if max_tokens >= ceiling:
            raise RuntimeError("Still truncated at the ceiling; split the task instead of accepting partial output")
        max_tokens = min(max_tokens * 4, ceiling)

full = create_complete(ask)
text = full.content[0].text.strip().removeprefix("```json").removesuffix("```").strip()
print("parsed items:", len(json.loads(text)))

## You should now have
- [ ] A successful raw REST call and SDK call
- [ ] `config.py` with a pinned model ID and limits
- [ ] `log_response()` and a `Chat` helper that keeps history
- [ ] A truncation check that never accepts partial output

## Check yourself
1. Which HTTP header carries your API key, and where should its value come from?
2. Why did the follow-up fail without history?
3. What should your code do when `stop_reason` is `max_tokens`?